# Chatbot Agente con Skills (Herramientas, Memoria y RAG)

Este notebook toma la base de RAG y le suma capacidades de Agente usando LangChain ReAct.
El asistente puede decidir usar una Herramienta de Búsqueda Web (DuckDuckGo Search) o la Base de Conocimiento RAG (FAISS) según la pregunta del usuario.

In [14]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFDirectoryLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')
from langchain.agents import create_react_agent, AgentExecutor, Tool
from langchain.memory import ConversationBufferMemory
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.prompts import PromptTemplate


In [15]:
print("Cargando archivos TXT de la carpeta 'data/'...")
loader_txt = DirectoryLoader('data/', glob="**/*.txt", loader_cls=TextLoader)
documentos_txt = loader_txt.load()

print("Scrapeando datos abiertos del repositorio de la Universidad Nacional de Colombia...")
# URL publica de ejemplo en el Repositorio Institucional UN
url_repositorio = "https://repositorio.unal.edu.co/handle/10336/1"
loader_web = WebBaseLoader(url_repositorio)
documentos_web = loader_web.load()
print(f"Se obtuvieron {len(documentos_web)} documentos de la web.")

# Consolidar documentos
documentos = documentos_txt + documentos_web
print(f"En total, se tienen {len(documentos)} documentos para procesar exitosamente.")

Cargando archivos TXT de la carpeta 'data/'...
Scrapeando datos abiertos del repositorio de la Universidad Nacional de Colombia...
Se obtuvieron 1 documentos de la web.
En total, se tienen 1 documentos para procesar exitosamente.


In [16]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
textos_divididos = text_splitter.split_documents(documentos)
print(f"El texto fue subdividido en {len(textos_divididos)} fragmentos.")

El texto fue subdividido en 6 fragmentos.


In [17]:
# Embebidos 100% locales con all-MiniLM-L6-v2 (muy eficiente)
print("Descargando/Cargando modelo de embeddings en HuggingFace...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Creando y llenando la base de datos vectorial local (FAISS)...")
vectorstore = FAISS.from_documents(textos_divididos, embeddings)
print("Base creada correctamente!")

Descargando/Cargando modelo de embeddings en HuggingFace...
Creando y llenando la base de datos vectorial local (FAISS)...
Base creada correctamente!


In [18]:
# Cargamos nuevamente el modelo LLM ligero desde HF.
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"
model_path = hf_hub_download(repo_id=model_name_or_path, filename=model_basename)

llm = LlamaCpp(
    model_path=model_path,
    temperature=0.1,  # Temperatura baja para evitar alucinación RAG
    max_tokens=2048,
    top_p=1,
    verbose=False  # Para no llenar de logs la ejecución
)

In [19]:
!pip install -U duckduckgo-search

In [20]:
# Crear herramienta RAG
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

# Definir herramientas
search_tool = DuckDuckGoSearchRun()

tools = [
    Tool(
        name="Base_Conocimientos_UNAL",
        func=qa_chain.run,
        description="Útil para responder preguntas específicas sobre los datos, documentos y la Universidad Nacional proporcionados. Úsalo antes que la búsqueda web si es sobre la universidad."
    ),
    Tool(
        name="Busqueda_Web",
        func=search_tool.run,
        description="Útil para buscar en internet información reciente, eventos actuales, o cosas que no están en la Base de Conocimientos."
    )
]

# Plantilla del Prompt ReAct
template = """Responde a la pregunta del usuario lo mejor que puedas en un lenguaje sencillo y amigable. Tienes acceso a las siguientes herramientas (tools):

{tools}

Para usar una herramienta, usa EXACTAMENTE el siguiente formato:

Thought: Do I need to use a tool? Yes
Action: la accion a tomar, debe ser UNA de estas [{tool_names}]
Action Input: la entrada de busqueda a la accion
Observation: el resultado de la accion

Cuando ya sepas la respuesta final (o si no necesitas usar herramientas), usa este formato OBLIGATORIAMENTE:

Thought: Do I need to use a tool? No
Final Answer: [tu respuesta final al usuario en español]

¡Comienza! Recuerda siempre terminar con Final Answer: cuando tengas la respuesta.

Historial de conversación:
{chat_history}

Pregunta del usuario: {input}
{agent_scratchpad}"""

prompt_react = PromptTemplate.from_template(template)

# Crear Agente ReAct
agent = create_react_agent(llm, tools, prompt_react)

# Crear Agent Executor con memoria
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
agent_executor = AgentExecutor(agent=agent, tools=tools, memory=memory, verbose=True, handle_parsing_errors=True)



In [21]:
import gradio as gr

def responder_agente(mensaje, historial):
    try:
        # Pasa el input al AgentExecutor el cual maneja su propio loop ReAct
        respuesta = agent_executor.invoke({"input": mensaje})
        return respuesta.get("output", "No se resolvió la pregunta.")
    except Exception as e:
        return f"Ocurrió un error en el agente: {e}"

demo_agente = gr.ChatInterface(
    fn=responder_agente,
    title="Chatbot Agente Autonómo (Web + RAG)",
    description="Este agente decide si usar la Búsqueda Web o la Base de Conocimiento Local para responder a tus preguntas."
)

demo_agente.launch(share=True)



Running on local URL:  http://127.0.0.1:7862
IMPORTANT: You are using gradio version 4.25.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://8132e2494f7a816f3c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)




> Entering new AgentExecutor chain...
presentadas por los estudiantes de la Universidad Nacional de Colombia en el año 2020.

Thought: Do I need to use a tool? Yes
Action: Base_Conocimientos_UNAL
Action Input: tesis 2020 Universidad Nacional Colombia